## **Play with Images**

In [2]:
from ipywidgets import FileUpload
from IPython.display import display

uploader = FileUpload(accept=".png", multiple=False)
display(uploader)


FileUpload(value=(), accept='.png', description='Upload')

In [3]:
print(uploader.value)

({'name': '63298260369.png', 'type': 'image/png', 'size': 958023, 'content': <memory at 0x7a52f8237dc0>, 'last_modified': datetime.datetime(2025, 11, 15, 14, 50, 40, 475000, tzinfo=datetime.timezone.utc)},)


In [4]:
uploader.value[0]

{'name': '63298260369.png',
 'type': 'image/png',
 'size': 958023,
 'content': <memory at 0x7a52f8237dc0>,
 'last_modified': datetime.datetime(2025, 11, 15, 14, 50, 40, 475000, tzinfo=datetime.timezone.utc)}

## **Convert this is into the `base64` format.**

In [5]:
import base64

uploaded_file = uploader.value[0]

# the memory view
content_mv = uploaded_file["content"]

# convert memory view to bytes
img_bytes = bytes(content_mv)

# now base64 encode
img_b64 = base64.b64encode(img_bytes).decode("utf-8")

In [6]:
import os
from dotenv import load_dotenv
load_dotenv()


os.environ["OPENAI_API_KEY"]=os.getenv("OPENAI_API_KEY")

from langchain.chat_models import init_chat_model

model = init_chat_model("gpt-4.1-mini")

In [7]:
from langchain.agents import create_agent

agent = create_agent(
    model=model
)

In [8]:
from langchain.messages import HumanMessage

multimodel_questions = HumanMessage(content=[
    {"type": "text", "text": "Tell me about this image as its as he looks eyes and other details."},
    {"type": "image", "base64": img_b64, "mime_type": "image/png"}
])

response = agent.invoke(
    {"messages": [multimodel_questions]}
)

print(response["messages"][-1].content)

The image shows a man with medium brown skin and dark, wavy hair. His eyes are dark brown and he has a calm, composed expression. He has well-groomed facial hair, including a mustache and a short beard along his jawline and chin. He is wearing a dark suit jacket over a light blue or white dress shirt with the top button undone, giving a slightly relaxed look to his formal attire. The background is plain black, which makes him stand out prominently in the image. Overall, he looks professional and confident.


## **Play with audio**

In [30]:
import sounddevice as sd
from scipy.io.wavfile import write
import base64
import io
import time
from tqdm import tqdm

duration = 10
sample_rate = 44100

print("Recording...")
audio = sd.rec(
    int(duration * sample_rate),
    samplerate=sample_rate,
    channels=1
)

for _ in tqdm(range(duration * 10)):
    time.sleep(0.1)

sd.wait()
print("Done...")

# write WAV to an in-memory buffer
buf = io.BytesIO()
write(buf, sample_rate, audio)

wav_bytes = buf.getvalue()

# correct base64 encoding
aud_b64 = base64.b64encode(wav_bytes).decode("utf-8")


Recording...


100%|██████████| 100/100 [00:10<00:00,  9.82it/s]

Done...


In [33]:
from openai import OpenAI
import io

client = OpenAI()

audio_file = io.BytesIO(wav_bytes)
audio_file.name = "audio.wav"  # IMPORTANT: OpenAI requires a filename

transcript = client.audio.transcriptions.create(
    file=audio_file,
    prompt="What ever you hare just give the output as it is.",
    model="gpt-4o-transcribe"  # or "whisper-1"
)

text = transcript.text
print(text)


এইটা বাংলা ইনিশিয়েটিভ রেকর্ড করা হচ্ছে, বর্তমানে এটা মডেল টেস্ট করার জন্য।


In [15]:
audio_file = io.BytesIO(wav_bytes)
audio_file.name = "audio.wav"  # REQUIRED

transcript = client.audio.transcriptions.create(
    file=audio_file,
    model="whisper-1",
    response_format="text"  # optional but clean
)

text = transcript
print(text)


માંડાંપુંંતે બાઈજ્ રિકરકુરાઓચે માંડાંતે ટિસ્ટીંગ પરપાસ્ રિકરકુરાઓચે માંડાંતે ટિ�



In [17]:
transcript = client.audio.transcriptions.create(
    file=audio_file,
    model="whisper-1",
    prompt="This audio is in Bengali (Bangla).",
    response_format="text"
)

print(transcript)


এক মোছে voice record করারো চে মোঝে testing purpose recording করারো চে মোঝে

